In [1]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [2]:
documents[2]

{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
 'section': 'General course-related questions',
 'question': 'Course - Can I still join the course after the start date?',
 'course': 'data-engineering-zoomcamp'}

In [3]:
import minsearch

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [5]:
from groq import Groq

client = Groq(
    api_key="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",
)

In [6]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [7]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [11]:
def llm(prompt):
    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [12]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [13]:
rag('how do I run kafka?')

'To run Kafka, you have a few options depending on your setup:\n\n1. **Java Kafka**: In the project directory, run `java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java` in the terminal.\n\n2. **Python Kafka**: \n   - Create a virtual environment and install the required packages by running `python -m venv env`, then `source env/bin/activate` (or `env\\Scripts\\activate` on Windows), and finally `pip install -r ../requirements.txt`.\n   - Make sure Docker images are up and running before attempting to run the Python file.\n   - If you encounter a permission error with `build.sh`, run `chmod +x build.sh` in the terminal.\n\nNote: If you encounter a `ModuleNotFoundError` related to `kafka.vendor.six.moves`, consider using `kafka-python-ng` by running `pip install kafka-python-ng`.'

In [14]:
rag('the course has already started, can I still enroll?')

'Yes, you can still enroll in the course even if it has already started. According to the FAQ, "Yes, even if you don\'t register, you\'re still eligible to submit the homeworks." However, be aware that there will be deadlines for turning in the final projects, so it\'s recommended not to leave everything for the last minute.'

# RAG with Vector Search


In [15]:

from qdrant_client import QdrantClient, models


qd_client = QdrantClient("http://localhost:6333")


EMBEDDING_DIMENSIONALITY = 512
model_handle = "jinaai/jina-embeddings-v2-small-en"


collection_name = "zoomcamp-faq"

qd_client.delete_collection(collection_name=collection_name)





False

In [16]:

qd_client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY,
        distance=models.Distance.COSINE
    )
)

True

In [17]:

qd_client.create_payload_index(
    collection_name=collection_name,
    field_name="course",
    field_schema="keyword"
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [18]:

points = []

for i, doc in enumerate(documents):
    text = doc['question'] + ' ' + doc['text']
    vector = models.Document(text=text, model=model_handle)
    point = models.PointStruct(
        id=i,
        vector=vector,
        payload=doc
    )
    points.append(point)

In [19]:

qd_client.upsert(
    collection_name=collection_name,
    points=points
)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

onnx/model.onnx:   0%|          | 0.00/130M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [22]:
def vector_search(question):
    print('vector_search is used')
    
    course = 'data-engineering-zoomcamp'
    query_points = qd_client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=question,
            model=model_handle 
        ),
        query_filter=models.Filter( 
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=course)
                )
            ]
        ),
        limit=5,
        with_payload=True
    )
    
    results = []
    
    for point in query_points.points:
        results.append(point.payload)
    
    return results

In [21]:
def rag(query):
    search_results = vector_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [23]:

rag('how do I run kafka?')

vector_search is used


"To run Kafka, you need to follow these steps:\n\n1. Make sure the Kafka broker docker container is running. If it's not, use `docker ps` to confirm and then run `docker compose up -d` in the docker compose yaml file folder to start all instances.\n\n2. Create a virtual environment for Python files (if applicable) by running:\n   - `python -m venv env` (only once)\n   - `source env/bin/activate` (every time you need the virtual environment)\n\n3. Ensure the `StreamsConfig.BOOTSTRAP_SERVERS_CONFIG` is set to the correct server URL in your Java scripts (e.g., `JsonConsumer.java`, `JsonProducer.java`).\n\n4. Verify that the cluster key and secrets are updated in `Secrets.java` (e.g., `KAFKA_CLUSTER_KEY` and `KAFKA_CLUSTER_SECRET`).\n\n5. Run the Java Kafka scripts (e.g., `JsonProducer.java`, `JsonConsumer.java`) using the command:\n   - `java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java` (replace `<jar_name>` with your actual jar name)\n\nNote